# Prepare ANN Evaluation Datasets

This notebook filters `FINAL_MASTER.csv` into six scenario-specific datasets used by the four ANN evaluation scripts.

## Scenarios Overview

| File | Hour column | Description |
|------|-------------|-------------|
| `master_no_quiet.csv` | UTC | All rows **excluding** geomagnetically quiet days |
| `master_no_disturbed.csv` | UTC | All rows **excluding** geomagnetically disturbed days |
| `master_no_midday_midnight.csv` | Standard LT | All rows **excluding** local midday (LT 11–13) and midnight (LT 23, 0, 1) |
| `quiet_days_only.csv` | UTC | Rows on quiet days only |
| `disturbed_days_only.csv` | UTC | Rows on disturbed days only |
| `midday_midnight_only.csv` | Standard LT | Rows at local midday or midnight only |

Quiet/disturbed day classification comes from `quiet_and_disturbed.txt` (NOAA Kp-index format).

Local time for the midday/midnight scenario uses **fixed standard-time UTC offsets — DST is not applied**. The converted local hour replaces the UTC Hour column in `master_no_midday_midnight.csv` and `midday_midnight_only.csv` so that Scenario 4's ANN trains and predicts on local-time hours directly.

**Outputs land in:** `ann_evals/data/`

## 1. Imports and Path Setup

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date

In [ ]:
# Resolve paths relative to this notebook (ann_evals/notebooks/)
notebook_dir = Path().resolve()                 # ann_evals/notebooks/
project_root = notebook_dir.parent.parent       # project root
data_dir     = notebook_dir.parent / "data"     # ann_evals/data/

MASTER_CSV = project_root / "FINAL_MASTER.csv"
QD_TXT     = project_root / "quiet_and_disturbed.txt"

print("Project root :", project_root)
print("Data dir     :", data_dir)
print("Master CSV   :", MASTER_CSV.exists())
print("QD text file :", QD_TXT.exists())

## 2. Station Timezone Mapping

Each station's IANA timezone is used to convert UTC hour to civil local time.  
DST is automatically handled by the timezone database for 2019 dates.

| Station | Location | IANA Timezone |
|---------|----------|---------------|
| CP | Cachoeira Paulista, Brazil | `America/Sao_Paulo` |
| ElginAB | Eglin AFB, Florida, USA | `America/Chicago` |
| Jicamarca | Jicamarca, Peru | `America/Lima` |
| MilstonHill | Millstone Hill, Massachusetts, USA | `America/New_York` |
| Ramey | Ramey, Puerto Rico | `America/Puerto_Rico` |

In [ ]:
## 2. Station Standard-Time UTC Offsets

Local time for midday/midnight classification uses **fixed standard-time offsets — DST is not applied**.

| Station | Location | UTC Offset | Time Zone Name |
|---------|----------|------------|----------------|
| CP | Cachoeira Paulista, Brazil | UTC−3 | BRT (Brazil Standard Time) |
| ElginAB | Eglin AFB, Florida, USA | UTC−6 | CST (Central Standard Time) |
| Jicamarca | Jicamarca, Peru | UTC−5 | PET (Peru Time, no DST) |
| MilstonHill | Millstone Hill, Massachusetts, USA | UTC−5 | EST (Eastern Standard Time) |
| Ramey | Ramey, Puerto Rico | UTC−4 | AST (Atlantic Standard Time, no DST) |

Conversion: `local_hour = (utc_hour + offset) % 24`

STATION_UTC_OFFSET = {
    'CP':          -3,   # BRT  — Brazil Standard Time
    'ElginAB':     -6,   # CST  — Central Standard Time
    'Jicamarca':   -5,   # PET  — Peru Time (no DST)
    'MilstonHill': -5,   # EST  — Eastern Standard Time
    'Ramey':       -4,   # AST  — Atlantic Standard Time (Puerto Rico, no DST)
}

In [ ]:
def _parse_field(line: str, start: int):
    """Return int from a 2-char field at `start`, or None if blank."""
    chunk = line[start:start + 2] if len(line) >= start + 2 else "  "
    return int(chunk.strip()) if chunk.strip() else None


def parse_quiet_disturbed(path: Path):
    """
    Returns (quiet_doys, disturbed_doys) as sets of integer day-of-year values
    derived from the NOAA monthly quiet/disturbed day file.
    """
    quiet_doys: set[int] = set()
    disturbed_doys: set[int] = set()

    with open(path, "r") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if len(line) < 7 or not line[:4].strip().isdigit():
                continue  # skip header / blank lines

            year  = int(line[0:4])
            month = int(line[5:7])

            quiet_days     = []
            disturbed_days = []

            for i in range(5):           # q1-q5  (positions 8-17)
                d = _parse_field(line, 8 + i * 2)
                if d:
                    quiet_days.append(d)
            for i in range(5):           # q6-q10 (positions 19-28)
                d = _parse_field(line, 19 + i * 2)
                if d:
                    quiet_days.append(d)
            for i in range(5):           # d1-d5  (positions 30-39)
                d = _parse_field(line, 30 + i * 2)
                if d:
                    disturbed_days.append(d)

            for day in quiet_days:
                try:
                    quiet_doys.add(date(year, month, day).timetuple().tm_yday)
                except ValueError:
                    pass

            for day in disturbed_days:
                try:
                    disturbed_doys.add(date(year, month, day).timetuple().tm_yday)
                except ValueError:
                    pass

    return quiet_doys, disturbed_doys

In [ ]:
## 5. Compute Standard Local Time

The `Hour` column in `FINAL_MASTER.csv` is in **UTC**. For Scenario 4 we need standard civil local time (no DST) at each station.

Approach: `local_hour = (utc_hour + offset) % 24`

The resulting `LT` column is:
- Used only to build midday/midnight masks
- Written into the `Hour` column of `master_no_midday_midnight.csv` and `midday_midnight_only.csv`
- **Not** written into the quiet/disturbed CSVs (those keep UTC Hour)

def compute_standard_lt(df: pd.DataFrame) -> pd.Series:
    """Convert UTC Hour to standard local time using fixed UTC offsets (no DST)."""
    lt = pd.Series(-1, index=df.index, dtype=int)
    for station, offset in STATION_UTC_OFFSET.items():
        mask = df["Station"] == station
        if mask.any():
            lt[mask] = (df.loc[mask, "Hour"].astype(int) + offset) % 24
    if (lt == -1).any():
        unknown = df.loc[lt == -1, "Station"].unique()
        raise ValueError(f"No UTC offset mapping for stations: {unknown}")
    return lt

In [ ]:
print("Computing standard local time (fixed offsets, no DST)...")
df["LT"] = compute_standard_lt(df)
print(f"LT range: {df['LT'].min()} – {df['LT'].max()}")
print()
print("LT distribution (all stations):")
df["LT"].value_counts().sort_index()

## 5. Compute Geopolitical Local Time

The `Hour` column in `FINAL_MASTER.csv` is in **UTC**. For Scenario 4 (excluding midday/midnight), we need civil local time at each station.

Approach:
1. Reconstruct an absolute UTC timestamp from `DayOfYear` + `Hour` anchored to 2019-01-01
2. Use `pandas` timezone conversion via the IANA database (handles DST automatically)
3. Extract the local hour and store it in a temporary `LT` column

The `LT` column is dropped before saving — it is not a model feature.

In [ ]:
def compute_geo_lt(df: pd.DataFrame) -> pd.Series:
    """Convert UT Hour to geopolitical civil local time for each row."""
    total_seconds = ((df["DayOfYear"].astype(int) - 1) * 24 + df["Hour"].astype(int)) * 3600
    utc_times = pd.to_datetime(total_seconds, unit="s", origin="2019-01-01", utc=True)

    lt = pd.Series(-1, index=df.index, dtype=int)
    for station, tz_name in STATION_TZ.items():
        mask = df["Station"] == station
        if mask.any():
            lt[mask] = utc_times[mask].dt.tz_convert(tz_name).dt.hour.values

    if (lt == -1).any():
        unknown = df.loc[lt == -1, "Station"].unique()
        raise ValueError(f"No timezone mapping for stations: {unknown}")
    return lt

In [ ]:
print("Computing geopolitical local time...")
df["LT"] = compute_geo_lt(df)
print(f"LT range: {df['LT'].min()} – {df['LT'].max()}")
print()
print("LT distribution (all stations):")
df["LT"].value_counts().sort_index()

data_dir.mkdir(parents=True, exist_ok=True)

# quiet/disturbed CSVs — keep UTC Hour
df_utc = df.drop(columns=["LT"])

# midday/midnight CSVs — replace Hour with standard local time
df_lt = df.copy()
df_lt["Hour"] = df_lt["LT"]
df_lt = df_lt.drop(columns=["LT"])

outputs = {
    "master_no_quiet.csv"           : (df_utc, ~mask_quiet),
    "master_no_disturbed.csv"       : (df_utc, ~mask_disturbed),
    "master_no_midday_midnight.csv" : (df_lt,  ~mask_mid),
    "quiet_days_only.csv"           : (df_utc,  mask_quiet),
    "disturbed_days_only.csv"       : (df_utc,  mask_disturbed),
    "midday_midnight_only.csv"      : (df_lt,   mask_mid),
}

for fname, (frame, mask) in outputs.items():
    subset = frame[mask]
    subset.to_csv(data_dir / fname, index=False)
    hour_type = "std LT" if "midday" in fname or "midnight" in fname else "UTC"
    print(f"  {fname:<40} {len(subset):>7,} rows  Hour={hour_type}  ->  saved")

print()
print("All datasets written to:", data_dir)

In [ ]:
MIDDAY_HOURS   = {11, 12, 13}
MIDNIGHT_HOURS = {23, 0, 1}

mask_quiet     = df["DayOfYear"].isin(quiet_doys)
mask_disturbed = df["DayOfYear"].isin(disturbed_doys)
mask_midday    = df["LT"].isin(MIDDAY_HOURS)
mask_midnight  = df["LT"].isin(MIDNIGHT_HOURS)
mask_mid       = mask_midday | mask_midnight

total = len(df)
print(f"Total rows            : {total:>7,}")
print(f"Quiet rows            : {mask_quiet.sum():>7,}  ({mask_quiet.mean()*100:.1f}%)")
print(f"Disturbed rows        : {mask_disturbed.sum():>7,}  ({mask_disturbed.mean()*100:.1f}%)")
print(f"Midday rows (LT)      : {mask_midday.sum():>7,}  ({mask_midday.mean()*100:.1f}%)")
print(f"Midnight rows (LT)    : {mask_midnight.sum():>7,}  ({mask_midnight.mean()*100:.1f}%)")
print(f"Midday+Midnight total : {mask_mid.sum():>7,}  ({mask_mid.mean()*100:.1f}%)")

## 7. Write Output Datasets

The temporary `LT` column is dropped before saving — it is only needed for mask construction, not model training.

In [ ]:
data_dir.mkdir(parents=True, exist_ok=True)
df_save = df.drop(columns=["LT"])

outputs = {
    "master_no_quiet.csv"           : df_save[~mask_quiet],
    "master_no_disturbed.csv"       : df_save[~mask_disturbed],
    "master_no_midday_midnight.csv" : df_save[~mask_mid],
    "quiet_days_only.csv"           : df_save[mask_quiet],
    "disturbed_days_only.csv"       : df_save[mask_disturbed],
    "midday_midnight_only.csv"      : df_save[mask_mid],
}

for fname, subset in outputs.items():
    path = data_dir / fname
    subset.to_csv(path, index=False)
    print(f"  {fname:<40} {len(subset):>7,} rows  ->  saved")

print()
print("All datasets written to:", data_dir)

## 8. Dataset Summary

Quick sanity check to verify row counts and column consistency across all written files.

In [ ]:
summary_rows = []
for fname in outputs:
    tmp = pd.read_csv(data_dir / fname)
    summary_rows.append({
        "File": fname,
        "Rows": len(tmp),
        "Columns": len(tmp.columns),
        "Stations": ", ".join(sorted(tmp["Station"].unique())),
        "DOY range": f"{tmp['DayOfYear'].min()}–{tmp['DayOfYear'].max()}",
    })

pd.DataFrame(summary_rows).set_index("File")